In [ ]:
# ============================================
# CELL 1 - Jokes Data
# ============================================
text = """why did the chicken cross the road? to get to the other side!
what do you call a fish without eyes? a fsh!
why dont scientists trust atoms? because they make up everything!
what do you call fake spaghetti? an impasta!
why did the scarecrow win an award? outstanding in his field!
what do you call cheese that isnt yours? nacho cheese!
what do you call a sleeping dinosaur? a dino snore!
why did the math book look sad? it had too many problems!
what do you call an alligator in a vest? an investigator!
knock knock whos there? lettuce. lettuce who? lettuce in its cold!
why do cows wear bells? because their horns dont work!
what did the ocean say to the beach? nothing it just waved!
why did the bicycle fall over? because it was two tired!
what do you call a bear with no teeth? a gummy bear!
why cant elsa have a balloon? she will let it go!
what do you call a pig that does karate? a pork chop!
why did the golfer bring extra pants? in case he got a hole in one!
what do you call a dinosaur that crashes their car? tyrannosaurus wrecks!
why did the banana go to the doctor? because it wasnt peeling well!
what do you call a snowman with a six pack? an abdominal snowman!"""

print("Total characters:", len(text))

# ============================================
# CELL 2 - Tokenizer
# ============================================
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print("Vocab size:", vocab_size)

# ============================================
# CELL 3 - Data Split
# ============================================
import torch
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print("Train:", len(train_data), "Val:", len(val_data))

# ============================================
# CELL 4 - Batches
# ============================================
block_size = 16
batch_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print("Shape:", xb.shape)
print("Example input :", decode(xb[0].tolist()))
print("Example output:", decode(yb[0].tolist()))

# ============================================
# CELL 5 - Build Model
# ============================================
import torch.nn as nn
import torch.nn.functional as F

class JokeModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64)
        self.linear1 = nn.Linear(64, 128)
        self.linear2 = nn.Linear(128, vocab_size)

    def forward(self, idx, targets=None):
        x = self.embedding(idx)
        x = x[:, -1, :]
        x = F.relu(self.linear1(x))
        logits = self.linear2(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits, targets[:, -1])
        return logits, loss

model = JokeModel()
print("✅ Model ready! Params:", sum(p.numel() for p in model.parameters()))

# ============================================
# CELL 6 - Train
# ============================================
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Training...\n")
for step in range(3000):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 300 == 0:
        bar = "█" * int((1 - loss.item()/4) * 20)
        print(f"Step {step:4d} | Loss: {loss.item():.3f} | [{bar:<20}]")

print("\n✅ Done training!")

# ============================================
# CELL 7 - Generate
# ============================================
def generate_joke(starter, length=100):
    result = starter
    context = torch.tensor([encode(starter)], dtype=torch.long)
    for _ in range(length):
        logits, _ = model(context)
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        next_char = itos[next_token.item()]
        result += next_char
        new_tok = torch.tensor([[next_token.item()]], dtype=torch.long)
        context = torch.cat([context, new_tok], dim=1)[:, -block_size:]
        if next_char in ['!', '?'] and len(result) > 20:
            break
    return result

print("🤖 Joke 1:", generate_joke("why "))
print("🤖 Joke 2:", generate_joke("what do you call "))
print("🤖 Joke 3:", generate_joke("knock knock "))

Total characters: 1174
Vocab size: 29
Train: 1056 Val: 118
Shape: torch.Size([8, 16])
Example input :  bicycle fall ov
Example output: bicycle fall ove
✅ Model ready! Params: 13917
Training...

Step    0 | Loss: 3.447 | [██                  ]
Step  300 | Loss: 2.515 | [███████             ]
Step  600 | Loss: 1.536 | [████████████        ]
Step  900 | Loss: 2.395 | [████████            ]
Step 1200 | Loss: 2.026 | [█████████           ]
Step 1500 | Loss: 2.399 | [████████            ]
Step 1800 | Loss: 2.279 | [████████            ]
Step 2100 | Loss: 1.842 | [██████████          ]
Step 2400 | Loss: 2.340 | [████████            ]
Step 2700 | Loss: 2.099 | [█████████           ]

✅ Done training!
🤖 Joke 1: why ol cal pan kesnyo heyou thas fe wong y agoshoopakea jus slo athes ceichakepall ken hadou n sat ndora
🤖 Joke 2: what do you call car to bin t the!
🤖 Joke 3: knock knock helessallees heo jema at wheaset thall ng wa to aling toalel do?


In [ ]:
# ============================================
# CELL 1 - Install
# ============================================
!pip install transformers datasets torch -q

# ============================================
# CELL 2 - Dataset
# ============================================
texts = [
    "why did the chicken cross the road? to get to the other side!",
    "what do you call a fish without eyes? a fsh!",
    "why dont scientists trust atoms? they make up everything!",
    "what do you call fake spaghetti? an impasta!",
    "why did the scarecrow win an award? outstanding in his field!",
    "what do you call a sleeping dinosaur? a dino snore!",
    "why did the math book look sad? too many problems!",
    "knock knock whos there? lettuce. lettuce in its cold!",
    "the weather today is very sunny and warm",
    "i went to the store to buy some groceries",
    "python is a great programming language",
    "the meeting is scheduled for tomorrow at 9am",
    "i need to finish my homework tonight",
    "the cat is sleeping on the couch",
    "today i learned how to cook pasta",
    "the movie starts at 8pm tonight",
]
labels = [1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0]
print("✅ Dataset ready!")

# ============================================
# CELL 3 - Tokenize
# ============================================
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
encodings = tokenizer(texts, truncation=True, padding=True,
                      max_length=64, return_tensors="pt")
print("✅ Tokenized! Shape:", encodings['input_ids'].shape)

# ============================================
# CELL 4 - Dataset Class
# ============================================
import torch
from torch.utils.data import Dataset

class JokeDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

dataset = JokeDataset(encodings, labels)
print("✅ Dataset size:", len(dataset))

# ============================================
# CELL 5 - Train
# ============================================
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="./joke-classifier",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    logging_steps=5,
    save_strategy="no"
)

trainer = Trainer(model=model, args=training_args, train_dataset=dataset)
print("🏋️ Training...")
trainer.train()
print("✅ Done!")

# ============================================
# CELL 6 - Test
# ============================================
from transformers import pipeline

classifier = pipeline("text-classification",
                      model=model, tokenizer=tokenizer)

test_sentences = [
    "why did the bicycle fall over? it was two tired!",
    "i am going to the gym tomorrow morning",
    "what do you call a bear with no teeth? a gummy bear!",
    "the traffic today was really bad",
]

print("🤖 Joke Detector:\n")
for sentence in test_sentences:
    result = classifier(sentence)[0]
    label = "😂 JOKE" if result['label'] == 'LABEL_1' else "😐 NOT A JOKE"
    print(f"{label} ({result['score']:.0%})")
    print(f"  → '{sentence}'\n")

✅ Dataset ready!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🏋️ Training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
5,0.671728
10,0.544786
15,0.407919
20,0.339330


✅ Done!
🤖 Joke Detector:

😂 JOKE (89%)
  → 'why did the bicycle fall over? it was two tired!'

😐 NOT A JOKE (57%)
  → 'i am going to the gym tomorrow morning'

😂 JOKE (89%)
  → 'what do you call a bear with no teeth? a gummy bear!'

😐 NOT A JOKE (55%)
  → 'the traffic today was really bad'



In [4]:
# ============================================
# CELL 1 - Install
# ============================================
!pip install groq gradio pypdf -q
print("✅ Done!")

# ============================================
# CELL 2 - Full App
# ============================================
import gradio as gr
import pypdf
from groq import Groq

client = Groq(api_key="paste-your-api-key")
pdf_text = ""

def load_pdf(file):
    global pdf_text
    if file is None:
        return " No file uploaded"
    reader = pypdf.PdfReader(file)
    pdf_text = ""
    for page in reader.pages:
        pdf_text += page.extract_text()
    return f" PDF loaded! {len(reader.pages)} pages ready."

def ask_pdf(message, history):
    global pdf_text
    if not pdf_text:
        return " Please upload a PDF first!"
    messages = [
        {
            "role": "system",
            "content": f"Answer using ONLY this document:\n{pdf_text[:50000]}"
        }
    ]
    for human, bot in history:
        messages.append({"role": "user", "content": human})
        messages.append({"role": "assistant", "content": bot})
    messages.append({"role": "user", "content": message})
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages
    )
    return response.choices[0].message.content

with gr.Blocks() as demo:
    gr.Markdown("# PDF Q&A Bot")
    gr.Markdown("### Upload any PDF and ask questions!")
    pdf_upload = gr.File(label=" Upload PDF", file_types=[".pdf"])
    status = gr.Textbox(label="Status", interactive=False,
                        value="No PDF loaded yet")
    pdf_upload.change(fn=load_pdf, inputs=pdf_upload, outputs=status)
    gr.Markdown("###  Ask questions")
    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder="Ask anything about your PDF...")
    clear = gr.Button("Clear chat")

    def respond(message, chat_history):
        if chat_history is None:
            chat_history = []
        bot_message = ask_pdf(message, chat_history)
        chat_history = chat_history + [(message, bot_message)]
        return "", chat_history

    msg.submit(respond, inputs=[msg, chatbot], outputs=[msg, chatbot])
    clear.click(lambda: None, None, chatbot)

demo.launch(share=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 8.3 MB/s eta 0:00:00
✅ Done!


/tmp/ipykernel_12081/2602771487.py:55: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()
/tmp/ipykernel_12081/2602771487.py:55: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot()


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5642c1343423f5a9ab.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
